# 第 1 周第 1 天 —— 英语时态老师

## 练习目标（理念）

做一个小型「英语时态」学习助手：用 **Chat Completions API** 串起三步流水线——

1. **生成**指定时态、主题的句子（学生母语）
2. **翻译**成目标英语时态（本练习用便宜模型模拟学生翻译）
3. **批改**学生译文（语法、词汇、时态）

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `messages`（system / user） | 生成 / 翻译 / 批改各有一套 system + user |
| Chat Completions API | `openai.chat.completions.create(...)` |
| 换模型试效果 | `model_override` 切换 `gpt-5-mini` / `gpt-5-nano` |

## 怎么跑

1. 准备 `.env`：`OPENAI_API_KEY`
2. 从上到下依次运行单元格
3. 在参数格改 `language` / `tense` / `theme` 再重跑后续格


In [ ]:
# ========== 导入与环境：把密钥读进进程，再创建 OpenAI 客户端 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI

# 加载 .env；override=True 表示用 .env 覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量取出 API Key（后面创建客户端时 SDK 也会自动读 OPENAI_API_KEY）
api_key = os.getenv("OPENAI_API_KEY")
# 创建默认 OpenAI 客户端（无参时会使用环境变量里的密钥与默认 base_url）
openai = OpenAI()


In [ ]:
# ========== 封装一次 Chat Completions 调用 ==========

# 定义函数：把 messages 发给模型，返回助手回复的纯文本
def callGptMini(messages, model_override=None):
    # 默认模型 id（字符串必须与账号可用模型一致；不要改成别的名字除非你本机已开通）
    # 型号 =“gpt-4o-mini”
    model = "gpt-5-mini"

    # 若调用方传入 model_override，则改用该模型（例如更便宜的 nano）
    if model_override is not None:
        model = model_override

    # 发起非流式 chat.completions：一次拿完整回复
    response = openai.chat.completions.create(model=model, messages=messages)
    # choices[0].message.content 是助手文本；strip() 去掉首尾空白
    return response.choices[0].message.content.strip()


## 句子生成指令

下面两个函数分别拼 **system** 与 **user** 提示词：告诉模型「你是英语老师」以及「语言 / 时态 / 主题」。  
发给模型的英文 prompt **保持原样**（改译会改变生成行为）。


In [ ]:
# ========== 拼装「生成句子」用的 system / user 提示词 ==========

# 生成侧 system：规定老师角色、句数、时态与主题约束（f-string 把句数嵌进英文指令）
def composeSystemSentenceGenerationInstructions(number_of_sentences=3):
    return f"""You are an English teacher who helps students improve their english.

Your task is to generate {number_of_sentences} sentence in user's language.
The sentences should be in the specified tense and follow the given theme.
The sentences should be of varying complexity, from simple to complex.
"""


# 生成侧 user：把语言、时态、主题三项参数填进模板
def composeUserSentenceGenerationInstructions(language, tense, theme="any"):
    return f"""
  Language: {language}
  Tense: {tense}
  Theme: {theme}
"""


# 打印一条示例 user 指令，方便肉眼检查模板是否拼对
print(
    "Example:",
    composeUserSentenceGenerationInstructions("pl", "present perfect", "travel"),
)


## 定义一组时态 + 主题参数

真实产品里这些应由用户输入；这里先写死，方便一步步跑通流水线。


In [ ]:
# ========== 练习参数：语言 / 时态 / 主题（后续提示词都会读这些变量）==========

# 应该通过用户输入来获得
# 学生母语代码（示例：波兰语 pl）
language = "pl"
# 目标英语时态名称（会原样写进 prompt）
tense = "past perfect"
# 句子主题
theme = "travel"


In [ ]:
# ========== 组装「生成句子」的 messages 列表（system + user）==========

# Chat Completions 需要的 messages：角色 + content
sentence_generation_messages = [
    # system：怎么生成、生成几句
    {"role": "system", "content": composeSystemSentenceGenerationInstructions(3)},
    {
        # user：本次的语言、时态、主题
        "role": "user",
        "content": composeUserSentenceGenerationInstructions(language, tense, theme),
    },
]


## 生成要翻译的句子

调用默认模型，得到学生母语下的练习句（稍后当作「原文」）。


In [ ]:
# ========== 调用 API：生成待翻译句子并打印 ==========

# 用默认 gpt-5-mini 生成句子文本
sentences_to_translate = callGptMini(sentence_generation_messages)
# 打印模型返回，便于人工检查
print(sentences_to_translate)
# 如果需要的话，我们可以将句子分成一个列表


## 模拟学生翻译

用更便宜的模型（`gpt-5-nano`）扮演「学生」，把原文译成指定时态的英语，供下一步批改。


In [ ]:
# ========== 翻译阶段：拼 prompt → 调便宜模型 → 打印「模拟学生译文」==========

# 拼 user 侧翻译指令：原文、源语言、目标语言、要求的时态
def composeUserTranslationInstructions(
    sentences, source_language, target_language, tense
):
    return f"""
  Translate the following sentences from {source_language} to {target_language}.
  The translations should use {tense} tense.

  Sentences:
  {sentences}
  """


# messages：system 定「准确翻译」角色；user 放具体待译内容
translation_messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant that translates sentences accurately.",
    },
    {
        "role": "user",
        "content": composeUserTranslationInstructions(
            sentences_to_translate, language, "english", tense
        ),
    },
]

# model_override 换成更便宜的模型，模拟「学生」水平/成本更低的回答
translations = callGptMini(translation_messages, model_override="gpt-5-nano")

# 打印模拟译文，供下一步批改使用
print("Simulated translations:\n", translations)


## 创建批改用提示词

教师角色的 system 会强调「必须检查指定时态」；user 则提交学生译文。


In [ ]:
# ========== 批改阶段：拼 system / user 校验提示词（英文指令保留）==========

# system：英语老师审阅语法、词汇，并特别盯紧指定时态
def composeValidationSystemPrompt(user_language, tense, original_sentences):
    return f"""You are an English teacher who reviews students' sentences for grammar and vocabulary accuracy.
User's task was to translate sentences from {user_language} into English using {tense} tense.

Your task is to review the translations provided by the student.

# 指南
	- The sentences are specifically meant to be in {tense} tense.
  - For each sentence, provide feedback on grammar and vocabulary.
  - If the sentence is correct, respond with "Correct" and optionally provide some feedback.
  - If there are mistakes, provide the corrected sentence along with an explanation of the errors.
  - Keep your feedback concise and focused on the most important issues.
  - Verb tense usage is especially important; ensure the translations correctly use {tense} tense.

The original sentences were:
{original_sentences}

Now, analyze and review the user's translations.
"""


# user：把学生译文塞进模板
def composeValidationUserPrompt(user_translations):
    return f"""
  My translations:
  {user_translations}
  """


# 打印两条示例，确认模板占位符替换正确
print(
    "Example:",
    composeValidationSystemPrompt("pl", tense, "sentence1\nsentence2\nsentence3"),
)
print(
    "Example:",
    composeValidationUserPrompt("My translation1\nMy translation2\nMy translation3"),
)


## 评估学生译文

把「原文 + 模拟译文」交给批改模型，打印反馈。


In [ ]:
# ========== 调用批改模型：输出 Validation feedback ==========

# 组装批改 messages：system 含原文与时态要求；user 含学生译文
validation_messages = [
    {
        "role": "system",
        "content": composeValidationSystemPrompt(
            language, tense, sentences_to_translate
        ),
    },
    {"role": "user", "content": composeValidationUserPrompt(translations)},
]

# 用 gpt-5-mini 做批改（比 nano 更稳的审阅质量）
validation_feedback = callGptMini(validation_messages, model_override="gpt-5-mini")
# 打印批改结果（前缀文案保留原样）
print("🗒️ Validation feedback:\n", validation_feedback)


## 总结与可改进点

当前工作流已经跑通：**生成 → 模拟翻译 → 批改**。

后续可改进：

- **强制严格格式**：生成句编号 1…X，每句独占一行，响应里不要夹杂其它文字
- **语言参数化**：把「正在学习的语言」当作显式参数传入
- **白名单校验**：对照模型支持的语言列表，拒绝未知语言代码
